# LogFormer — Log Anomaly Detection Benchmark

**Paper:** Guo et al. (AAAI 2024) — *LogFormer: A Pre-train and Tuning Pipeline for Log Anomaly Detection*  
**Architecture:** Transformer Encoder + Adapter, pre-trained then tuned  
**Datasets:** HDFS_v1 · BGL  

---
### Kaggle Setup
1. Upload dataset to Kaggle Dataset:
   - `HDFS.log` → `/kaggle/input/logad-data/hdfs/HDFS.log`
   - `anomaly_label.csv` → `/kaggle/input/logad-data/hdfs/anomaly_label.csv`
   - `BGL.log` → `/kaggle/input/logad-data/bgl/BGL.log`
2. **Enable GPU** (T4 or P100 recommended)
3. Run All

**Pipeline:**  
`Raw log → Drain parse → SentenceTransformer embed → window sequences → Transformer train → Adapter tune → Evaluate`

In [ ]:
!pip install -q sentence-transformers drain3 tqdm

In [ ]:
import os, sys, time, json, random, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Paths ─────────────────────────────────────────────────────────────────────
HDFS_LOG    = '/kaggle/input/logad-data/hdfs/HDFS.log'
HDFS_LABEL  = '/kaggle/input/logad-data/hdfs/anomaly_label.csv'
BGL_LOG     = '/kaggle/input/logad-data/bgl/BGL.log'
OUTPUT_DIR  = '/kaggle/working/logformer_results'
CKPT_DIR    = '/kaggle/working/logformer_checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ── Hyperparams ───────────────────────────────────────────────────────────────
EMBEDDING_DIM = 768
WINDOW_SIZE   = 20
BATCH_SIZE    = 64
PRE_EPOCHS    = 5     # pre-training epochs (increase for better results)
TUNE_EPOCHS   = 10    # adapter tuning epochs
LR_PRETRAIN   = 1e-5
LR_TUNE       = 1e-5
ADAPTER_SIZE  = 64
NUM_LAYERS    = 1
print('Config OK')

## 1. Log Parsing with Drain

In [ ]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

def parse_with_drain(log_path, dataset_name, max_lines=None):
    """Parse raw log → (EventTemplate, Label) records."""
    config = TemplateMinerConfig()
    # tune for each dataset
    if dataset_name == 'HDFS':
        config.drain_sim_th = 0.5
        config.drain_depth  = 4
    else:  # BGL
        config.drain_sim_th = 0.5
        config.drain_depth  = 4

    miner = TemplateMiner(config=config)
    records = []

    with open(log_path, 'rb') as f:
        for i, raw in enumerate(f):
            if max_lines and i >= max_lines:
                break
            if i % 500_000 == 0 and i > 0:
                print(f'  Drain {dataset_name}: {i:,} lines')
            try:
                line = raw.decode('utf-8').strip()
            except:
                continue

            if dataset_name == 'HDFS':
                parts = line.split(' ', 5)
                if len(parts) < 6:
                    continue
                content = parts[5]
                # extract block ID for grouping
                blk_id = None
                for tok in content.split():
                    if tok.startswith('blk_'):
                        blk_id = tok.strip('.')
                        break
                result = miner.add_log_message(content)
                records.append({'BlockId': blk_id, 'EventTemplate': result['template_mined']})

            else:  # BGL
                parts = line.split(' ')
                if len(parts) < 9:
                    continue
                label   = parts[0]
                content = ' '.join(parts[8:])
                result  = miner.add_log_message(content)
                records.append({'Label': label, 'EventTemplate': result['template_mined']})

    df = pd.DataFrame(records)
    print(f'{dataset_name}: {len(df):,} rows, {df["EventTemplate"].nunique()} templates')
    return df


print('Parsing HDFS...')
df_hdfs = parse_with_drain(HDFS_LOG, 'HDFS')
# merge with labels
label_df = pd.read_csv(HDFS_LABEL)
label_dict = dict(zip(label_df['BlockId'], label_df['Label'].map({'Normal': 'Normal', 'Anomaly': 'Anomaly'})))
df_hdfs['Label'] = df_hdfs['BlockId'].map(label_dict).fillna('Normal')

print('\nParsing BGL...')
df_bgl = parse_with_drain(BGL_LOG, 'BGL')

## 2. Sentence Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

print('Loading SentenceTransformer (distilbert-base-nli-mean-tokens)...')
embedder = SentenceTransformer('distilbert-base-nli-mean-tokens', device=str(DEVICE))

def embed_templates(df):
    """Embed unique EventTemplates then map back to each row."""
    unique_templates = df['EventTemplate'].unique().tolist()
    print(f'Embedding {len(unique_templates)} unique templates...')
    embeddings = embedder.encode(unique_templates, batch_size=256,
                                 show_progress_bar=True, convert_to_numpy=True)
    t2vec = dict(zip(unique_templates, embeddings))
    df = df.copy()
    df['Vector'] = df['EventTemplate'].map(t2vec)
    return df

print('\nEmbedding HDFS templates...')
df_hdfs = embed_templates(df_hdfs)
print('\nEmbedding BGL templates...')
df_bgl  = embed_templates(df_bgl)

## 3. Window Sequences

In [ ]:
def make_hdfs_windows(df, train_ratio=0.8):
    """HDFS: group by BlockId → one sequence per block."""
    x_data, y_data, block_ids = [], [], []
    for blk, grp in tqdm(df.groupby('BlockId'), desc='HDFS windows'):
        vecs = np.array(grp['Vector'].tolist(), dtype=np.float32)
        label = int(grp.iloc[0]['Label'] == 'Anomaly')
        y = [0, 0]; y[label] = 1
        x_data.append(vecs)
        y_data.append(y)
        block_ids.append(blk)
    n = len(x_data)
    t = int(train_ratio * n)
    return (x_data[:t], y_data[:t]), (x_data[t:], y_data[t:])


def make_bgl_windows(df, window_size=20, train_ratio=0.8):
    """BGL: fixed sliding window (non-overlapping)."""
    x_data, y_data = [], []
    vecs   = df['Vector'].tolist()
    labels = df['Label'].tolist()
    for i in range(0, len(vecs) - window_size + 1, window_size):
        chunk_v = np.array(vecs[i:i+window_size], dtype=np.float32)
        chunk_l = labels[i:i+window_size]
        is_anom = int(any(l != '-' for l in chunk_l))
        y = [0, 0]; y[is_anom] = 1
        x_data.append(chunk_v)
        y_data.append(y)
    n = len(x_data)
    t = int(train_ratio * n)
    return (x_data[:t], y_data[:t]), (x_data[t:], y_data[t:])


print('Creating HDFS windows...')
hdfs_train, hdfs_test = make_hdfs_windows(df_hdfs)
print(f'HDFS train: {len(hdfs_train[0])}, test: {len(hdfs_test[0])}')

print('\nCreating BGL windows...')
bgl_train, bgl_test = make_bgl_windows(df_bgl, window_size=WINDOW_SIZE)
print(f'BGL train: {len(bgl_train[0])}, test: {len(bgl_test[0])}')

## 4. Model Definition

In [ ]:
class LogDataset(Dataset):
    def __init__(self, x, y, window_size=WINDOW_SIZE):
        self.x = x
        self.y = y
        self.window_size = window_size

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        xi = self.x[idx]
        pad = np.zeros((self.window_size, EMBEDDING_DIM), dtype=np.float32)
        n = min(len(xi), self.window_size)
        pad[:n] = xi[:n]
        return pad, np.array(self.y[idx], dtype=np.float32)


class Adapter(nn.Module):
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)
        self.gelu = nn.GELU()

    def forward(self, x):
        return x + self.fc2(self.gelu(self.fc1(x)))


class LogFormerEncoder(nn.Module):
    def __init__(self, d_model=768, nhead=8, adapter_size=64,
                 num_layers=1, window_size=WINDOW_SIZE, mode='classifier'):
        super().__init__()
        self.mode = mode
        # Learned positional encoding
        self.pos_emb = nn.Embedding(window_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=4*d_model, dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        if mode in ('adapter', 'classifier'):
            self.adapter = Adapter(d_model, adapter_size)

        self.classifier = nn.Linear(d_model, 2)

    def forward(self, x):
        # x: (B, W, D)
        B, W, D = x.shape
        pos = torch.arange(W, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos)
        x = self.encoder(x)           # (B, W, D)
        x = x.mean(dim=1)             # pool
        if self.mode in ('adapter', 'classifier'):
            x = self.adapter(x)
        return self.classifier(x)     # (B, 2)


print('Model classes defined.')

## 5. Training Helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total_loss, preds_all, true_all = 0.0, [], []
    criterion = nn.CrossEntropyLoss()
    for x, y in tqdm(loader, desc=f'Train epoch {epoch}', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds_all.extend(out.argmax(1).cpu().tolist())
        true_all.extend(y.argmax(1).cpu().tolist())
    if scheduler:
        scheduler.step()
    f1 = f1_score(true_all, preds_all, zero_division=0)
    return total_loss / len(loader), f1


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    preds_all, true_all = [], []
    for x, y in tqdm(loader, desc='Eval', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out  = model(x.float())
        preds_all.extend(out.argmax(1).cpu().tolist())
        true_all.extend(y.argmax(1).cpu().tolist())
    p, r, f, _ = precision_recall_fscore_support(
        true_all, preds_all, average='binary', zero_division=0)
    return dict(precision=float(p), recall=float(r), f1=float(f))


def run_experiment(train_data, test_data, dataset_name,
                   pre_epochs=PRE_EPOCHS, tune_epochs=TUNE_EPOCHS):
    x_tr, y_tr = train_data
    x_te, y_te = test_data
    ws = max(len(x) for x in x_tr[:100])  # estimate max window size
    ws = min(ws, WINDOW_SIZE)

    train_ds = LogDataset(x_tr, y_tr, window_size=ws)
    test_ds  = LogDataset(x_te, y_te, window_size=ws)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    history = {'pretrain_loss': [], 'pretrain_f1': [],
               'tune_loss': [],     'tune_f1': [],
               'eval_pretrain': {}, 'eval_tune': {}}

    # ── Phase 1: Pre-training (classifier mode) ───────────────────────────────
    print(f'\n=== {dataset_name}: Pre-training ({pre_epochs} epochs) ===')
    model = LogFormerEncoder(mode='classifier', window_size=ws).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=LR_PRETRAIN)
    sched = optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    best_f1, best_ckpt = 0, None

    for epoch in range(1, pre_epochs + 1):
        loss, f1 = train_one_epoch(model, train_ld, opt, sched, epoch)
        metrics  = evaluate_model(model, test_ld)
        history['pretrain_loss'].append(loss)
        history['pretrain_f1'].append(metrics['f1'])
        print(f'  Epoch {epoch}/{pre_epochs} | loss={loss:.4f} | '
              f'P={metrics["precision"]:.4f} R={metrics["recall"]:.4f} F1={metrics["f1"]:.4f}')
        if metrics['f1'] >= best_f1:
            best_f1   = metrics['f1']
            best_ckpt = {k: v.clone() for k, v in model.state_dict().items()}

    history['eval_pretrain'] = evaluate_model(model, test_ld)
    ckpt_path = os.path.join(CKPT_DIR, f'{dataset_name}_pretrain.pt')
    torch.save({'model': best_ckpt}, ckpt_path)
    print(f'Pre-train best F1: {best_f1:.4f} | saved: {ckpt_path}')

    # ── Phase 2: Adapter tuning ───────────────────────────────────────────────
    print(f'\n=== {dataset_name}: Adapter tuning ({tune_epochs} epochs) ===')
    model_tune = LogFormerEncoder(mode='adapter', window_size=ws).to(DEVICE)
    model_tune.load_state_dict(best_ckpt, strict=False)

    # freeze encoder, only train adapter + classifier
    for name, param in model_tune.named_parameters():
        param.requires_grad = ('adapter' in name or 'classifier' in name)

    opt_tune   = optim.Adam(
        filter(lambda p: p.requires_grad, model_tune.parameters()), lr=LR_TUNE)
    sched_tune = optim.lr_scheduler.StepLR(opt_tune, step_size=5, gamma=0.5)
    best_f1_tune = 0

    for epoch in range(1, tune_epochs + 1):
        loss, f1 = train_one_epoch(model_tune, train_ld, opt_tune, sched_tune, epoch)
        metrics  = evaluate_model(model_tune, test_ld)
        history['tune_loss'].append(loss)
        history['tune_f1'].append(metrics['f1'])
        print(f'  Epoch {epoch}/{tune_epochs} | loss={loss:.4f} | '
              f'P={metrics["precision"]:.4f} R={metrics["recall"]:.4f} F1={metrics["f1"]:.4f}')
        if metrics['f1'] >= best_f1_tune:
            best_f1_tune = metrics['f1']
            torch.save({'model': model_tune.state_dict()},
                       os.path.join(CKPT_DIR, f'{dataset_name}_tuned_best.pt'))

    history['eval_tune'] = evaluate_model(model_tune, test_ld)
    print(f'Tune best F1: {best_f1_tune:.4f}')
    return history

print('Training helpers ready.')

## 6. Run — HDFS

In [ ]:
hist_hdfs = run_experiment(hdfs_train, hdfs_test, 'HDFS',
                           pre_epochs=PRE_EPOCHS, tune_epochs=TUNE_EPOCHS)

## 7. Run — BGL

In [ ]:
hist_bgl = run_experiment(bgl_train, bgl_test, 'BGL',
                          pre_epochs=PRE_EPOCHS, tune_epochs=TUNE_EPOCHS)

## 8. Save Results & Visualize

In [ ]:
results = {
    'metadata': {'model': 'LogFormer', 'paper': 'AAAI 2024',
                 'seed': SEED, 'pre_epochs': PRE_EPOCHS, 'tune_epochs': TUNE_EPOCHS},
    'hdfs': {
        'pretrain': hist_hdfs['eval_pretrain'],
        'tuned':    hist_hdfs['eval_tune'],
    },
    'bgl': {
        'pretrain': hist_bgl['eval_pretrain'],
        'tuned':    hist_bgl['eval_tune'],
    },
}
out_path = os.path.join(OUTPUT_DIR, 'logformer_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('LogFormer Training Curves', fontsize=14, fontweight='bold')

for i, (hist, dname) in enumerate([(hist_hdfs, 'HDFS'), (hist_bgl, 'BGL')]):
    # Pre-train
    ax = axes[i][0]
    ax.plot(hist['pretrain_loss'], label='Loss',  color='#F44336')
    ax2 = ax.twinx()
    ax2.plot(hist['pretrain_f1'], label='F1', color='#2196F3', linestyle='--')
    ax.set_title(f'{dname} — Pre-training', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='#F44336')
    ax2.set_ylabel('F1', color='#2196F3')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')
    ax.grid(alpha=0.3)

    # Adapter tuning
    ax = axes[i][1]
    ax.plot(hist['tune_loss'], label='Loss',  color='#FF9800')
    ax2 = ax.twinx()
    ax2.plot(hist['tune_f1'], label='F1', color='#4CAF50', linestyle='--')
    ax.set_title(f'{dname} — Adapter Tuning', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='#FF9800')
    ax2.set_ylabel('F1', color='#4CAF50')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'logformer_curves.png'), bbox_inches='tight')
plt.show()
print('Done!')